# 04 · Hippo / YAP–TAZ Scores (Beginner‑Friendly)

**Goal:** compute per‑sample scores for Hippo/YAP gene sets from VST matrices and make simple plots.
Run after 03.


In [ ]:

BASE = "/content/drive/MyDrive/Colorectal_Hippo_Dysbiosis"
print("Project base:", BASE)


In [ ]:

import pandas as pd, numpy as np, json, matplotlib.pyplot as plt, re
from pathlib import Path

def signed_z_score(expr_df, pos_genes, neg_genes=None):
    g = [g for g in pos_genes if g in expr_df.index]
    X = expr_df.loc[g]
    Z = (X - X.mean(axis=1).values[:,None]) / X.std(axis=1, ddof=0).values[:,None]
    score = Z.mean(axis=0)
    if neg_genes:
        n = [g for g in neg_genes if g in expr_df.index]
        if len(n):
            Xn = expr_df.loc[n]
            Zn = (Xn - Xn.mean(axis=1).values[:,None]) / Xn.std(axis=1, ddof=0).values[:,None]
            score = score - Zn.mean(axis=0)
    return score


In [ ]:

# Load gene sets
with open(f"{BASE}/config/gene_sets.json") as fh:
    GENES = json.load(fh)
list(GENES.keys())


In [ ]:

# Load VST matrices if present
tcga_p = f"{BASE}/data_processed/tcga_vst.csv"
ibd_p  = f"{BASE}/data_processed/ibd_vst.csv"
tcga = pd.read_csv(tcga_p, index_col=0) if Path(tcga_p).exists() else None
ibd  = pd.read_csv(ibd_p,  index_col=0) if Path(ibd_p).exists() else None
print("TCGA VST:", None if tcga is None else tcga.shape, "| IBD VST:", None if ibd is None else ibd.shape)


In [ ]:

# Compute scores
if tcga is not None:
    yap = signed_z_score(tcga, GENES["YAP_TARGETS"]).rename("yap_score")
    hip = signed_z_score(tcga, GENES["HIPPO_CORE"]).rename("hippo_core_score")
    eff = signed_z_score(tcga, GENES["YAP_TAZ_EFFECTORS"]).rename("yap_taz_effectors_score")
    df = pd.concat([yap, hip, eff], axis=1)
    df.to_csv(f"{BASE}/results/hippo/tcga_scores.csv")
    print("saved -> results/hippo/tcga_scores.csv")

if ibd is not None:
    yap = signed_z_score(ibd, GENES["YAP_TARGETS"]).rename("yap_score")
    hip = signed_z_score(ibd, GENES["HIPPO_CORE"]).rename("hippo_core_score")
    eff = signed_z_score(ibd, GENES["YAP_TAZ_EFFECTORS"]).rename("yap_taz_effectors_score")
    df = pd.concat([yap, hip, eff], axis=1)
    df.to_csv(f"{BASE}/results/hippo/ibd_scores.csv")
    print("saved -> results/hippo/ibd_scores.csv")


In [ ]:

# Quick plots if metadata exists
try:
    # TCGA: try to detect tumor/normal from clinical
    man = pd.read_csv(f"{BASE}/config/manifest_tcga.csv").set_index("key")["path"].to_dict()
    clinical = pd.read_csv(man["clinical"])
    tcga_scores = pd.read_csv(f"{BASE}/results/hippo/tcga_scores.csv", index_col=0)

    # guess id and group
    cand_id = [c for c in clinical.columns if re.search("sample|barcode|submitter", c, flags=re.I)]
    id_col = cand_id[0] if cand_id else clinical.columns[0]
    tcols = [c for c in clinical.columns if re.search("tumor|normal|sample_type", c, flags=re.I)]
    if tcols:
        tcol = tcols[0]
        clinical[id_col] = clinical[id_col].astype(str).str.replace(r'[^A-Za-z0-9_\-\.]+','_', regex=True)
        tcga_scores.index = tcga_scores.index.astype(str).str.replace(r'[^A-Za-z0-9_\-\.]+','_', regex=True)
        dfp = tcga_scores.merge(clinical[[id_col,tcol]], left_index=True, right_on=id_col, how="inner")
        if dfp.shape[0] > 0:
            plt.figure(figsize=(5,4)); dfp.boxplot(by=tcol, column="yap_score", rot=45)
            plt.title("TCGA YAP score by tumor/normal"); plt.suptitle(""); plt.tight_layout(); plt.show()
except Exception as e:
    print("NOTE:", e)


In [ ]:

try:
    meta_p = f"{BASE}/data_processed/ibd_metadata_clean.csv"
    if Path(meta_p).exists():
        meta = pd.read_csv(meta_p)
        ibd_scores = pd.read_csv(f"{BASE}/results/hippo/ibd_scores.csv", index_col=0)
        cand_id = [c for c in meta.columns if re.search("sample|run|gsm|id", c, flags=re.I)]
        sid = cand_id[0] if cand_id else meta.columns[0]
        cand_group = [c for c in meta.columns if c.lower() in ["group","status","condition","phenotype","disease","diagnosis"]]
        if cand_group:
            gcol = cand_group[0]
            dfp = ibd_scores.reset_index().rename(columns={"index":"sample_id"}).merge(
                meta[[sid,gcol]], left_on="sample_id", right_on=sid, how="inner"
            )
            if dfp.shape[0] > 0:
                plt.figure(figsize=(5,4)); dfp.boxplot(by=gcol, column="yap_score", rot=45)
                plt.title("IBD YAP score by group"); plt.suptitle(""); plt.tight_layout(); plt.show()
except Exception as e:
    print("NOTE:", e)
